In [1]:
import polars as pl
from Python import statcast, batter_features

years = [2025]

In [2]:
print(batter_features.BUILD_COLUMNS)
print(statcast.DISCIPLINE_COUNTS)
print(statcast.STRIKEOUT_EVENTS)
print(statcast.NON_PA_EVENTS)

('game_pk', 'game_date', 'batter', 'stand', 'p_throws', 'home_team', 'away_team', 'inning_topbot', 'at_bat_number', 'events', 'description', 'type', 'zone', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom')
('InZone', 'OutZone', 'Swings', 'Chases', 'ZSwings', 'Contacts', 'ZContacts', 'OContacts')
frozenset({'strikeout', 'strikeout_double_play'})
frozenset({'pickoff_caught_stealing_home', 'stolen_base_home', 'runner_double_play', 'caught_stealing_home', 'wild_pitch', 'pickoff_caught_stealing_3b', 'pickoff_1b', 'caught_stealing_2b', 'pickoff_3b', 'stolen_base_3b', 'other_out', 'caught_stealing_3b', 'pickoff_caught_stealing_2b', 'passed_ball', 'stolen_base_2b', 'pickoff_2b'})


In [3]:
cross_check_extra = ["at_bat_number", "pitch_number", "inning"]
load_columns = list(dict.fromkeys(list(batter_features.BUILD_COLUMNS) + cross_check_extra))

raw = statcast.load_statcast_years(years, columns=load_columns)
raw.shape, raw.columns

((712528, 18),
 ['game_pk',
  'game_date',
  'batter',
  'stand',
  'p_throws',
  'home_team',
  'away_team',
  'inning_topbot',
  'at_bat_number',
  'events',
  'description',
  'type',
  'zone',
  'estimated_woba_using_speedangle',
  'woba_value',
  'woba_denom',
  'pitch_number',
  'inning'])

In [4]:
flagged = statcast.add_plate_discipline_flags(statcast.add_event_flags(raw))
flagged.select([
    "events", "description", "is_pa", "is_k", "is_bb", "is_hbp", "is_hr",
    "is_hit", "is_whiff", "is_called_strike",
    "is_in_zone", "is_out_zone", "is_swing", "is_contact", "is_chase",
]).head(10)

# Sanity: a strikeout must be a PA; a whiff must never also be scored as contact
flagged.select(
    (pl.col("is_k") & ~pl.col("is_pa")).sum().alias("k_without_pa"),
    (pl.col("is_whiff") & pl.col("is_contact")).sum().alias("whiff_and_contact"),
)

k_without_pa,whiff_and_contact
u32,u32
0,0


In [5]:
bat_team_check = flagged.with_columns(
    pl.when(pl.col("inning_topbot") == "Top")
      .then(pl.col("away_team"))
      .otherwise(pl.col("home_team"))
      .alias("bat_team")
)

# bat_team should always equal home_team or away_team, never null
bat_team_check.select(
    ((pl.col("bat_team") != pl.col("home_team")) & (pl.col("bat_team") != pl.col("away_team"))).sum().alias("bad_bat_team"),
    pl.col("bat_team").is_null().sum().alias("null_bat_team"),
)

bad_bat_team,null_bat_team
u32,u32
0,0


In [6]:
batter_games = batter_features.build_batter_games(raw)
batter_games.shape

# Uniqueness check on the primary key
batter_games.filter(pl.col("PA") == 0).select(["game_pk", "batter", "wOBA", "xwOBA"])

game_pk,batter,wOBA,xwOBA
i64,i64,f64,f64
778346,642350,NaN,NaN
778236,672515,NaN,NaN
777690,670764,NaN,NaN
776997,696285,NaN,NaN


In [7]:
checks = batter_games.select(
    (pl.col("PA_vL") + pl.col("PA_vR") == pl.col("PA")).all().alias("PA_split_matches_PA"),
    (pl.col("K_vL") + pl.col("K_vR") == pl.col("K")).all().alias("K_split_matches_K"),
    (pl.col("K") <= pl.col("PA")).all().alias("K_le_PA"),
    (pl.col("Chases") <= pl.col("OutZone")).all().alias("Chases_le_OutZone"),
    (pl.col("Contacts") <= pl.col("Swings")).all().alias("Contacts_le_Swings"),
    (pl.col("CSW") <= pl.col("Pitches")).all().alias("CSW_le_Pitches"),
)
checks

PA_split_matches_PA,K_split_matches_K,K_le_PA,Chases_le_OutZone,Contacts_le_Swings,CSW_le_Pitches
bool,bool,bool,bool,bool,bool
true,true,true,true,true,true


In [8]:
pa = statcast.plate_appearances(raw)
pa_check = (
    pa.group_by(["game_pk", "batter"])
    .agg(pl.len().alias("PA_check"), pl.col("is_k").sum().alias("K_check"))
)

merged = batter_games.join(pa_check, on=["game_pk", "batter"], how="left")
merged.select(
    (pl.col("PA") == pl.col("PA_check")).all().alias("PA_matches_plate_appearances"),
    (pl.col("K") == pl.col("K_check")).all().alias("K_matches_plate_appearances"),
)

PA_matches_plate_appearances,K_matches_plate_appearances
bool,bool
true,true


In [9]:
zero_pa = batter_games.filter(pl.col("PA") == 0)
zero_pa.height
zero_pa.select(["game_pk", "batter", "game_date", "Pitches", "BIP", "PA", "K"]).head(20)

game_pk,batter,game_date,Pitches,BIP,PA,K
i64,i64,date,u32,u32,u32,u32
778346,642350,2025-04-12,1,0,0,0
778236,672515,2025-04-20,2,0,0,0
777690,670764,2025-06-01,1,0,0,0
776997,696285,2025-07-26,2,0,0,0


In [10]:
rows_check = raw.filter(
    (pl.col("game_pk") == 745051) & (pl.col("batter") == 606466)
)
rows_check.select(["at_bat_number", "pitch_number", "description", "events", "type"])

at_bat_number,pitch_number,description,events,type
i64,i64,str,str,str


In [11]:
raw.filter((pl.col("game_pk") == 745051) & (pl.col("at_bat_number").is_between(68, 72))).select(
    ["batter", "at_bat_number", "pitch_number", "description", "events", "inning", "inning_topbot"]
).sort(["at_bat_number", "pitch_number"])

batter,at_bat_number,pitch_number,description,events,inning,inning_topbot
i64,i64,i64,str,str,i64,str


In [12]:
sample_batter = batter_games["batter"][0]

batter_games.filter(pl.col("batter") == sample_batter).select([
    "game_pk", "game_date", "bat_team", "opp_team", "is_home",
    "PA", "K", "BB", "HBP", "HR", "Hits",
    "PA_vL", "K_vL", "PA_vR", "K_vR",
    "Swings", "Chases", "Contacts",
    "chase_rate", "contact_rate", "swstr_rate",
    "xwOBA", "wOBA",
])

game_pk,game_date,bat_team,opp_team,is_home,PA,K,BB,HBP,HR,Hits,PA_vL,K_vL,PA_vR,K_vR,Swings,Chases,Contacts,chase_rate,contact_rate,swstr_rate,xwOBA,wOBA
i64,date,str,str,bool,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64
778563,2025-03-18,"""CHC""","""LAD""",true,1,0,0,0,0,0,1,0,0,0,3,1,3,0.333333,1.0,0.0,0.209,0.0
778564,2025-03-19,"""CHC""","""LAD""",true,1,0,1,0,0,0,1,0,0,0,1,0,1,0.0,1.0,0.0,0.691497,0.7
778515,2025-03-30,"""CHC""","""AZ""",false,3,0,1,0,0,0,2,0,1,0,6,2,5,0.2,0.833333,0.0625,0.275166,0.233333
778494,2025-04-01,"""CHC""","""ATH""",false,2,0,0,0,0,1,1,0,1,0,4,1,4,0.25,1.0,0.0,0.096,0.45
778471,2025-04-02,"""CHC""","""ATH""",false,5,1,0,0,0,1,3,1,2,0,15,6,13,0.375,0.866667,0.066667,0.491309,0.18
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
776267,2025-09-19,"""CHC""","""CIN""",false,3,2,0,0,0,1,3,2,0,0,9,6,4,0.666667,0.444444,0.384615,0.186667,0.3
776239,2025-09-21,"""CHC""","""CIN""",false,3,0,0,0,0,0,2,0,1,0,6,1,6,0.25,1.0,0.0,0.208667,0.0
776209,2025-09-23,"""CHC""","""NYM""",true,4,1,1,0,0,1,1,0,3,1,8,2,7,0.25,0.875,0.058824,0.302374,0.4
